# Modeling a Feedback System

## What is a Feedback System?

An open-loop system applies a pre-planned input regardless of what the plant actually does. A **feedback system** is different: it measures the plant's output, compares it to a desired reference, and adjusts the input to drive the error to zero.

The canonical closed-loop structure is:

```
r_k ──►[ controller ]──► τ_k ──►[ plant ]──► θ_k
           ▲                                   │
           └───────────────────────────────────┘
```

In `dynamicalnodes`, both blocks are `DynamicalSystem` objects. Composing them in a simulation loop — where the plant's output becomes the controller's input and vice versa — is all that is needed to close the loop.

![Closed-loop block diagram: reference r_k enters a sum junction producing error e_k, which feeds the P Controller to produce torque τ_k, which drives the Pendulum Plant producing angle θ_k fed back negatively to the sum junction.](../../_static/figures/feedback_pendulum.svg)

## The System

We use the ZOH-discretized pendulum from [the previous notebook](plant_with_input.ipynb):

$$x_{k+1} = F_{\text{ZOH}}\, x_k + G_{\text{ZOH}}\, \tau_k, \qquad y_k = \theta_k$$

The **controller** is a proportional (P) controller. It is stateless — no `f`, only `h` — and produces a corrective torque proportional to the angle error:

$$\tau_k = K_P\,(r_k - \theta_k)$$

The reference $r_k$ is a constant target angle.

## Implementation

In [ ]:
import numpy as np
from scipy.linalg import expm
from dynamicalnodes import DynamicalSystem

### ZOH Matrices

Computed identically to the previous notebook — same parameters, same augmented-exponential method.

In [ ]:
g = 9.81  # m/s²
L = 1.0   # m
m = 1.0   # kg
c = 0.3   # damping coefficient (1/s)
dt = 0.15 # s

A = np.array([[0.0,    1.0],
              [-g / L, -c]])
B = np.array([0.0, 1.0 / (m * L**2)])

M = np.zeros((3, 3))
M[:2, :2] = A
M[:2,  2] = B
M_exp = expm(M * dt)
F_zoh = M_exp[:2, :2]
G_zoh = M_exp[:2,  2]

### Plant Block

In [ ]:
def pendulum_f(xk, uk, F, G):
    return F @ xk + G * uk


def pendulum_h(xk):
    return xk[0]  # angle θ


plant = DynamicalSystem(f=pendulum_f, h=pendulum_h)

### Controller Block

The P controller is stateless — `h` maps the angle error directly to a corrective torque.

In [ ]:
def controller_h(rk, yk, KP):
    return KP * (rk - yk)


controller = DynamicalSystem(h=controller_h)

### Closing the Loop

On each step:
1. The controller reads `yk` (current angle) and `rk` (target angle), and produces torque `uk`.
2. The plant receives `uk`, advances its state, and returns the next angle `yk`.

In [ ]:
KP = 20.0  # N·m/rad
rk = 0.4   # rad  (~23° set-point)

sim_time = np.arange(0, 20, dt)

xk = np.zeros(2)
yk = 0.0   # bootstrap: angle before first plant step

ref_out, theta_out, torque_out, error_out = [], [], [], []

for _ in sim_time:
    uk = controller.step(rk=rk, yk=yk, KP=KP)
    xk, yk = plant.step(xk=xk, uk=uk, F=F_zoh, G=G_zoh)

    ref_out.append(rk)
    theta_out.append(yk)
    torque_out.append(uk)
    error_out.append(rk - yk)

### Results

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(3, 1, sharex=True, figsize=(10, 8))

axs[0].plot(sim_time, ref_out,   color="orange",    linestyle="--", linewidth=1.5, label="Reference $r_k$")
axs[0].plot(sim_time, theta_out, color="steelblue", linewidth=1.5,               label="Angle $\\theta_k$")
axs[0].set_ylabel("Angle (rad)")
axs[0].set_title(f"P-Controlled Pendulum — Angle Tracking  ($K_P = {KP}$, $r_k = {rk}$ rad)")
axs[0].legend()

axs[1].plot(sim_time, torque_out, color="tomato", linewidth=1.5)
axs[1].axhline(0, color="gray", linewidth=0.6, linestyle=":")
axs[1].set_ylabel("Torque $\\tau_k$ (N·m)")
axs[1].set_title("P Controller Output")

axs[2].plot(sim_time, error_out, color="purple", linewidth=1.5)
axs[2].axhline(0, color="gray", linewidth=0.6, linestyle=":")
axs[2].set_ylabel("Error $e_k = r_k - \\theta_k$ (rad)")
axs[2].set_xlabel("Time (s)")
axs[2].set_title("Tracking Error")

plt.tight_layout()
plt.show()

## Reading the Plot

The pendulum tracks the 0.4 rad set-point but oscillates around it before settling. This is expected: a P controller increases the effective stiffness of the plant but cannot increase the physical damping $c$. The closed-loop poles satisfy:

$$s^2 + c\,s + \left(\frac{g}{L} + \frac{K_P}{mL^2}\right) = 0$$

Increasing $K_P$ raises the natural frequency but leaves the damping ratio $\zeta = c / (2\omega_n)$ even smaller — making the oscillations faster but not shorter. A **PD controller** (adding a derivative term $K_D\dot{e}_k$) would inject synthetic damping and suppress the overshoot. A full **PID controller** would additionally eliminate any steady-state offset caused by model mismatch.

The [Cruise Control tutorial](../0_cruise_control/0_cruise_control.ipynb) shows this natural progression: PID controller + Kalman filter observer, both implemented as `DynamicalSystem` blocks composed in the same loop structure used here.

## Summary

A feedback system in `dynamicalnodes` is two `DynamicalSystem` blocks composed in a loop:

| Block | `f` | `h` | Role |
|---|---|---|---|
| Plant (ZOH) | $F_{\text{ZOH}} x_k + G_{\text{ZOH}} \tau_k$ | $\theta_k$ | linearized pendulum |
| P Controller | *(stateless)* | $K_P(r_k - \theta_k)$ | error-to-torque mapping |

The framework imposes no special structure for feedback — the loop is expressed naturally in Python and scales to any number of blocks.